# L1 — Rapport de Profiling : UCI Online Retail Dataset
## Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce
---
**Objectif** : Analyser exhaustivement la qualité du dataset principal UCI Online Retail.  
**Livrables** : Taux de nullité, outliers, doublons, distributions, anomalies métier pour 100% des colonnes.

> **Note** : Ce notebook utilise `ydata-profiling` pour le rapport automatique et des analyses custom Pandas/Plotly.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Installation (à exécuter une seule fois)
import subprocess
subprocess.run(['pip', 'install', 'ydata-profiling', 'plotly', 'seaborn', '-q'])
print(' Librairies installées')

 Librairies installées


In [3]:
import os

base = "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce"
print(os.listdir(base))

['Profiling', 'Data', 'Notebooks', 'DBT_Gold', 'ETL_Bronze_Silver']


In [4]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data"):
    for file in files:
        if "online" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver/online_retail_silver.parquet
/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv


## 0. Configuration & Imports

In [6]:
# ── Installation des dépendances (à exécuter une seule fois) ──
# !pip install ydata-profiling pandas plotly seaborn openpyxl

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
from IPython.display import display, HTML

# ── Paramètres globaux ──
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

DATASET_PATH = "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv"
import os
print(os.path.exists(DATASET_PATH))  # ← adapter au chemin réel dans votre Drive
REPORT_OUTPUT = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Profiling/L1_UCI_profile_report.html'

PALETTE = px.colors.qualitative.Set2

print('Imports OK')

True
Imports OK


## 1. Chargement du Dataset

In [7]:
# ── Chargement robuste avec gestion d'encodage ──
try:
    df_raw = pd.read_csv(
        DATASET_PATH,
        encoding='utf-8',
        parse_dates=['InvoiceDate'],
        dtype={
            'InvoiceNo': str,
            'StockCode': str,
            'Description': str,
            'CustomerID': str,
            'Country': str
        }
    )
    print(f' Chargement UTF-8 réussi : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
except UnicodeDecodeError:
    df_raw = pd.read_csv(
        DATASET_PATH,
        encoding='latin-1',
        parse_dates=['InvoiceDate'],
        dtype={
            'InvoiceNo': str,
            'StockCode': str,
            'Description': str,
            'CustomerID': str,
            'Country': str
        }
    )
    print(f' Chargement latin-1 (fallback) : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')

# Snapshot initial
df_raw.head(5)

 Chargement latin-1 (fallback) : 541,909 lignes × 8 colonnes


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.5500,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.7500,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom


## 2. Vue d'ensemble — Métriques Globales

In [9]:
# ── Métriques globales du dataset ──
total_rows = len(df_raw)
total_cols = df_raw.shape[1]
total_cells = total_rows * total_cols
total_nulls = df_raw.isnull().sum().sum()
total_dupes = df_raw.duplicated().sum()
returns_mask = df_raw['InvoiceNo'].str.startswith('C', na=False)
n_returns = returns_mask.sum()
n_sales = (~returns_mask).sum()

summary = {
    'Total lignes'           : f'{total_rows:,}',
    'Total colonnes'         : total_cols,
    'Taux nullité global'    : f'{total_nulls/total_cells*100:.2f}%',
    'Lignes dupliquées'      : f'{total_dupes:,} ({total_dupes/total_rows*100:.2f}%)',
    'Factures ventes'        : f'{n_sales:,} ({n_sales/total_rows*100:.1f}%)',
    'Factures retours (C*)'  : f'{n_returns:,} ({n_returns/total_rows*100:.1f}%)',
    'Clients uniques (non-null)' : df_raw['CustomerID'].nunique(),
    'SKUs uniques'           : df_raw['StockCode'].nunique(),
    'Pays uniques'           : df_raw['Country'].nunique(),
    'Période'                : f"{df_raw['InvoiceDate'].min().date()} → {df_raw['InvoiceDate'].max().date()}"
}

pd.DataFrame.from_dict(summary, orient='index', columns=['Valeur']).style.set_caption(' Vue Globale UCI Online Retail')

,Valeur
Total lignes,"541,909"
Total colonnes,8
Taux nullité global,3.15%
Lignes dupliquées,"5,268 (0.97%)"
Factures ventes,"532,621 (98.3%)"
Factures retours (C*),"9,288 (1.7%)"
Clients uniques (non-null),4372
SKUs uniques,4070
Pays uniques,38
Période,2010-12-01 → 2011-12-09


## 3. Analyse de Nullité — 100% des Colonnes

In [10]:
# ── Tableau de nullité par colonne ──
null_df = pd.DataFrame({
    'Colonne': df_raw.columns,
    'Type': df_raw.dtypes.values,
    'Nulls': df_raw.isnull().sum().values,
    'Taux_Null_%': (df_raw.isnull().sum() / len(df_raw) * 100).values,
    'Valeurs_uniques': df_raw.nunique().values,
    'Exemple_valeur': [df_raw[c].dropna().iloc[0] if df_raw[c].dropna().shape[0] > 0 else 'N/A' for c in df_raw.columns]
})

null_df['Criticité'] = null_df['Taux_Null_%'].apply(
    lambda x: 'Critique (>20%)' if x > 20 else ('Élevé (5-20%)' if x > 5 else (' Modéré (1-5%)' if x > 1 else ' OK (<1%)'))
)

display(null_df.style
    .format({'Taux_Null_%': '{:.2f}%'})
    .background_gradient(subset=['Taux_Null_%'], cmap='RdYlGn_r')
    .set_caption(' Analyse Nullité — Toutes les colonnes'))

# Visualisation
fig = px.bar(
    null_df.sort_values('Taux_Null_%', ascending=True),
    x='Taux_Null_%', y='Colonne', orientation='h',
    color='Taux_Null_%', color_continuous_scale='RdYlGn_r',
    title='Taux de Nullité par Colonne — UCI Online Retail',
    labels={'Taux_Null_%': 'Taux de nullité (%)'}
)
fig.add_vline(x=5, line_dash='dash', line_color='orange', annotation_text='Seuil 5%')
fig.add_vline(x=20, line_dash='dash', line_color='red', annotation_text='Seuil 20%')
fig.update_layout(height=400, showlegend=False)
fig.show()

,Colonne,Type,Nulls,Taux_Null_%,Valeurs_uniques,Exemple_valeur,Criticité
0,InvoiceNo,object,0,0.00%,25900,536365,OK (<1%)
1,StockCode,object,0,0.00%,4070,85123A,OK (<1%)
2,Description,object,1454,0.27%,4223,WHITE HANGING HEART T-LIGHT HOLDER,OK (<1%)
3,Quantity,int64,0,0.00%,722,6,OK (<1%)
4,InvoiceDate,datetime64[ns],0,0.00%,23260,2010-12-01 08:26:00,OK (<1%)
5,UnitPrice,float64,0,0.00%,1630,2.550000,OK (<1%)
6,CustomerID,object,135080,24.93%,4372,17850,Critique (>20%)
7,Country,object,0,0.00%,38,United Kingdom,OK (<1%)


## 4. Analyse Colonne par Colonne

### 4.1 — InvoiceNo

In [ ]:
col = 'InvoiceNo'
s = df_raw[col]

print(f'=== {col} ===\n')
print(f'  Nulls         : {s.isnull().sum():,} ({s.isnull().mean()*100:.2f}%)')
print(f'  Valeurs uniq. : {s.nunique():,}')
print(f'  Retours (C*)  : {s.str.startswith("C", na=False).sum():,}')
print(f'  Longueur min/max : {s.dropna().str.len().min()} / {s.dropna().str.len().max()}')
print(f'  Exemples : {s.dropna().unique()[:5]}')

# Distribution retours vs ventes
counts = pd.Series({'Ventes': (~s.str.startswith('C', na=False)).sum(),
                    'Retours (C*)': s.str.startswith('C', na=False).sum()})
fig = px.pie(values=counts.values, names=counts.index,
             title='InvoiceNo — Répartition Ventes vs Retours',
             color_discrete_sequence=PALETTE)
fig.show()

=== InvoiceNo ===

  Nulls         : 0 (0.00%)
  Valeurs uniq. : 25,900
  Retours (C*)  : 9,288
  Longueur min/max : 6 / 7
  Exemples : ['536365' '536366' '536367' '536368' '536369']


### 4.2 — StockCode

In [ ]:
col = 'StockCode'
s = df_raw[col]

# Codes spéciaux à exclure
SPECIAL_CODES = ['POST', 'D', 'C2', 'M', 'BANK CHARGES', 'AMAZONFEE', 'DOT', 'CRUK']
special_mask = s.isin(SPECIAL_CODES)

print(f'=== {col} ===')
print(f'  Nulls         : {s.isnull().sum():,}')
print(f'  Valeurs uniq. : {s.nunique():,}')
print(f'  Codes spéciaux à exclure : {special_mask.sum():,}')
print(f'  Détail codes spéciaux :')
display(s[special_mask].value_counts().rename_axis('StockCode').reset_index(name='Count'))

# Top 20 SKUs par fréquence
top_skus = s[~special_mask].value_counts().head(20)
fig = px.bar(x=top_skus.values, y=top_skus.index, orientation='h',
             title='Top 20 StockCode par fréquence d\'apparition',
             labels={'x': 'Occurrences', 'y': 'StockCode'})
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

=== StockCode ===
  Nulls         : 0
  Valeurs uniq. : 4,070
  Codes spéciaux à exclure : 2,845
  Détail codes spéciaux :


,StockCode,Count
0,POST,1256
1,DOT,710
2,M,571
3,C2,144
4,D,77
5,BANK CHARGES,37
6,AMAZONFEE,34
7,CRUK,16


### 4.3 — Description

In [11]:
col = 'Description'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls         : {s.isnull().sum():,} ({s.isnull().mean()*100:.2f}%)')
print(f'  Valeurs uniq. : {s.nunique():,}')
print(f'  Avec caractères spéciaux : {s.str.contains(r"[^a-zA-Z0-9 .,/-]", na=False).sum():,}')
print(f'  Descriptions en majuscules : {s.str.isupper().sum():,}')
print(f'  Longueur moy. : {s.dropna().str.len().mean():.1f} caractères')

# Inconsistances : même StockCode → descriptions différentes
desc_inconsistency = (
    df_raw.groupby('StockCode')['Description']
    .nunique()
    .reset_index()
    .rename(columns={'Description': 'nb_descriptions'})
    .query('nb_descriptions > 1')
)
print(f'\n   SKUs avec descriptions multiples : {len(desc_inconsistency):,}')
display(desc_inconsistency.head(10))

=== Description ===
  Nulls         : 1,454 (0.27%)
  Valeurs uniq. : 4,223
  Avec caractères spéciaux : 18,775
  Descriptions en majuscules : 537,363
  Longueur moy. : 26.6 caractères

   SKUs avec descriptions multiples : 650


,StockCode,nb_descriptions
1,10080,2
8,10133,2
26,15058A,2
28,15058C,2
31,16008,2
41,16045,2
49,16156L,2
56,16162M,2
57,16168M,2
58,16169E,2


### 4.4 — Quantity

In [12]:
col = 'Quantity'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls         : {s.isnull().sum():,}')
print(f'  Min / Max     : {s.min():,} / {s.max():,}')
print(f'  Négatifs (retours) : {(s < 0).sum():,} ({(s < 0).mean()*100:.2f}%)')
print(f'  Zéros         : {(s == 0).sum():,}')
print(f'  Extrêmes > 10 000 : {(s > 10000).sum():,}')

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outliers = s[(s < q1 - 3*iqr) | (s > q3 + 3*iqr)]
print(f'  Outliers IQR×3 : {len(outliers):,}')
display(s.describe().to_frame().style.set_caption('Stats Quantity'))

# Distribution (sans les extrêmes pour la lisibilité)
s_trimmed = s[(s >= -100) & (s <= 200)]
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Distribution Quantity (tronquée)', 'Boxplot Quantity'])
fig.add_trace(go.Histogram(x=s_trimmed, nbinsx=80, name='Quantity', marker_color=PALETTE[0]), row=1, col=1)
fig.add_trace(go.Box(y=s, name='Boxplot', boxpoints='outliers', marker_color=PALETTE[1]), row=1, col=2)
fig.update_layout(title='Quantity — Distribution & Outliers', height=400, showlegend=False)
fig.show()

=== Quantity ===
  Nulls         : 0
  Min / Max     : -80,995 / 80,995
  Négatifs (retours) : 10,624 (1.96%)
  Zéros         : 0
  Extrêmes > 10 000 : 3
  Outliers IQR×3 : 21,976


,Quantity
count,541909.000000
mean,9.552250
std,218.081158
min,-80995.000000
25%,1.000000
50%,3.000000
75%,10.000000
max,80995.000000


### 4.5 — InvoiceDate

In [13]:
col = 'InvoiceDate'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls         : {s.isnull().sum():,}')
print(f'  Type détecté  : {s.dtype}')
print(f'  Plage         : {s.min()} → {s.max()}')
print(f'  Durée couverte: {(s.max() - s.min()).days} jours')

# Transactions par mois
monthly = df_raw.set_index('InvoiceDate').resample('M')['InvoiceNo'].count().reset_index()
monthly.columns = ['Mois', 'Transactions']

fig = px.bar(monthly, x='Mois', y='Transactions',
             title='Volume de transactions par mois',
             color='Transactions', color_continuous_scale='Blues')
fig.update_layout(height=380)
fig.show()

# Heatmap heure × jour de semaine
df_raw['hour'] = df_raw['InvoiceDate'].dt.hour
df_raw['dayofweek'] = df_raw['InvoiceDate'].dt.day_name()
heatmap_data = df_raw.groupby(['dayofweek', 'hour'])['InvoiceNo'].count().unstack(fill_value=0)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = heatmap_data.reindex([d for d in day_order if d in heatmap_data.index])

fig2 = px.imshow(heatmap_data, color_continuous_scale='YlOrRd',
                 title='Heatmap — Transactions par jour et heure',
                 labels={'x': 'Heure', 'y': 'Jour', 'color': 'Nb transactions'})
fig2.update_layout(height=360)
fig2.show()

=== InvoiceDate ===
  Nulls         : 0
  Type détecté  : datetime64[ns]
  Plage         : 2010-12-01 08:26:00 → 2011-12-09 12:50:00
  Durée couverte: 373 jours


### 4.6 — UnitPrice

In [14]:
col = 'UnitPrice'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Prix = 0       : {(s == 0).sum():,} ({(s == 0).mean()*100:.2f}%)')
print(f'  Prix < 0       : {(s < 0).sum():,} (ajustements comptables)')
print(f'  Prix > 1 000   : {(s > 1000).sum():,}')
print(f'  Min / Max      : {s.min():.4f} / {s.max():.4f}')
print(f'  Médiane        : {s.median():.4f}')
display(s.describe().to_frame().style.format('{:.4f}').set_caption('Stats UnitPrice (GBP)'))

# Distribution
s_filtered = s[(s > 0) & (s < 50)]
fig = px.histogram(s_filtered, nbins=100,
                   title='UnitPrice — Distribution (0 < price < 50 GBP)',
                   labels={'value': 'Prix unitaire (GBP)'},
                   color_discrete_sequence=[PALETTE[2]])
fig.update_layout(height=380)
fig.show()

=== UnitPrice ===
  Nulls          : 0
  Prix = 0       : 2,515 (0.46%)
  Prix < 0       : 2 (ajustements comptables)
  Prix > 1 000   : 120
  Min / Max      : -11062.0600 / 38970.0000
  Médiane        : 2.0800


,UnitPrice
count,541909.0000
mean,4.6111
std,96.7599
min,-11062.0600
25%,1.2500
50%,2.0800
75%,4.1300
max,38970.0000


### 4.7 — CustomerID 🔴 (Colonne critique RFM/CLV)

In [15]:
col = 'CustomerID'
s = df_raw[col]

null_rate = s.isnull().mean() * 100

print(f'=== {col} ===  CRITIQUE POUR RFM & CLV ===')
print(f'  Nulls          : {s.isnull().sum():,} ({null_rate:.1f}%)')
print(f'  Clients uniques: {s.nunique():,}')
print(f'  Transactions avec client identifié : {s.notna().sum():,} ({s.notna().mean()*100:.1f}%)')

# Impact métier des nulls
null_revenue = df_raw.loc[s.isnull(), 'Quantity'] * df_raw.loc[s.isnull(), 'UnitPrice']
total_revenue = df_raw['Quantity'] * df_raw['UnitPrice']
revenue_anon = null_revenue.sum()
revenue_total = total_revenue.sum()

print(f'\n   CA transactions anonymes  : £{revenue_anon:,.0f}')
print(f'   CA total                  : £{revenue_total:,.0f}')
print(f'   Part CA anonyme           : {revenue_anon/revenue_total*100:.1f}%')

# Pie chart
fig = px.pie(
    values=[s.notna().sum(), s.isnull().sum()],
    names=['Identifié', 'Anonyme (NaN)'],
    title=f'CustomerID — {null_rate:.1f}% de transactions anonymes',
    color_discrete_sequence=[PALETTE[0], PALETTE[3]]
)
fig.show()

# Distribution des achats par client
orders_per_customer = df_raw.groupby('CustomerID')['InvoiceNo'].nunique()
fig2 = px.histogram(orders_per_customer, nbins=50,
                    title='Distribution — Nombre de commandes par client',
                    labels={'value': 'Nb commandes', 'count': 'Nb clients'},
                    color_discrete_sequence=[PALETTE[4]])
fig2.update_layout(height=380)
fig2.show()

=== CustomerID ===  CRITIQUE POUR RFM & CLV ===
  Nulls          : 135,080 (24.9%)
  Clients uniques: 4,372
  Transactions avec client identifié : 406,829 (75.1%)

   CA transactions anonymes  : £1,447,682
   CA total                  : £9,747,748
   Part CA anonyme           : 14.9%


### 4.8 — Country

In [ ]:
col = 'Country'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Pays uniques   : {s.nunique():,}')

country_counts = s.value_counts().reset_index()
country_counts.columns = ['Country', 'Count']
country_counts['Pct'] = country_counts['Count'] / len(s) * 100

print('\n  Top 10 pays :')
display(country_counts.head(10).style.format({'Pct': '{:.2f}%'}))

# Bar chart
fig = px.bar(country_counts.head(15), x='Country', y='Count',
             title='Top 15 pays — Volume de transactions',
             color='Count', color_continuous_scale='Blues',
             text='Count')
fig.update_layout(height=420, xaxis_tickangle=-45)
fig.show()

=== Country ===
  Nulls          : 0
  Pays uniques   : 38

  Top 10 pays :


,Country,Count,Pct
0,United Kingdom,495478,91.43%
1,Germany,9495,1.75%
2,France,8557,1.58%
3,EIRE,8196,1.51%
4,Spain,2533,0.47%
5,Netherlands,2371,0.44%
6,Belgium,2069,0.38%
7,Switzerland,2002,0.37%
8,Portugal,1519,0.28%
9,Australia,1259,0.23%


## 5. Détection des Doublons

In [ ]:
# ── Doublons stricts ──
dupes_strict = df_raw.duplicated()
print(f'Doublons stricts (toutes colonnes) : {dupes_strict.sum():,} ({dupes_strict.mean()*100:.3f}%)')

# ── Doublons partiels (même InvoiceNo + StockCode + Quantity) ──
dupes_partial = df_raw.duplicated(subset=['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice'])
print(f'Doublons partiels (Invoice+SKU+Qty+Price) : {dupes_partial.sum():,}')

if dupes_strict.sum() > 0:
    print('\nExemples de doublons stricts :')
    display(df_raw[dupes_strict].head(5))

Doublons stricts (toutes colonnes) : 5,268 (0.972%)
Doublons partiels (Invoice+SKU+Qty+Price) : 5,271

Exemples de doublons stricts :


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,hour,dayofweek
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.2500,17908,United Kingdom,11,Wednesday
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.1000,17908,United Kingdom,11,Wednesday
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.9500,17908,United Kingdom,11,Wednesday
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.9500,17908,United Kingdom,11,Wednesday
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.9500,17920,United Kingdom,11,Wednesday


## 6. Cohérence Métier & Anomalies

In [ ]:
# ── Calcul du Revenue ──
df_raw['Revenue'] = df_raw['Quantity'] * df_raw['UnitPrice']

print('=== Anomalies Métier ===')

# 1. Transactions avec prix nul ET quantité positive
zero_price_positive_qty = df_raw[(df_raw['UnitPrice'] == 0) & (df_raw['Quantity'] > 0)]
print(f'\n[1] Prix=0 & Qté>0       : {len(zero_price_positive_qty):,} lignes')

# 2. Quantité négative sans préfixe C (anomalie)
neg_qty_no_C = df_raw[(df_raw['Quantity'] < 0) & ~df_raw['InvoiceNo'].str.startswith('C', na=False)]
print(f'[2] Qté<0 sans facture C  : {len(neg_qty_no_C):,} lignes (incohérence)')

# 3. Factures C avec quantité positive
C_positive_qty = df_raw[df_raw['InvoiceNo'].str.startswith('C', na=False) & (df_raw['Quantity'] > 0)]
print(f'[3] Facture C & Qté>0     : {len(C_positive_qty):,} lignes (incohérence)')

# 4. Revenue total par type
df_raw['is_return'] = df_raw['InvoiceNo'].str.startswith('C', na=False)
rev_by_type = df_raw.groupby('is_return')['Revenue'].agg(['sum', 'mean', 'count'])
rev_by_type.index = ['Ventes', 'Retours']
print('\n[4] Revenue par type (ventes vs retours) :')
display(rev_by_type.style.format({'sum': '£{:,.0f}', 'mean': '£{:,.2f}', 'count': '{:,}'}))

# 5. Revenue CA net (ventes - retours)
ca_net = df_raw[~df_raw['is_return']]['Revenue'].sum() + df_raw[df_raw['is_return']]['Revenue'].sum()
print(f'\n[5] CA Net estimé (ventes + retours négatifs) : £{ca_net:,.0f}')

=== Anomalies Métier ===

[1] Prix=0 & Qté>0       : 1,179 lignes
[2] Qté<0 sans facture C  : 1,336 lignes (incohérence)
[3] Facture C & Qté>0     : 0 lignes (incohérence)

[4] Revenue par type (ventes vs retours) :


,sum,mean,count
Ventes,"£10,644,560",£19.99,"532,621"
Retours,"£-896,812",£-96.56,"9,288"



[5] CA Net estimé (ventes + retours négatifs) : £9,747,748


## 7. Rapport ydata-profiling (Automatique)

In [16]:
# ── Génération du rapport de profiling automatique ──
#  Ce rapport peut prendre 2-5 minutes sur le dataset complet

print(' Génération du rapport ydata-profiling...')

profile = ProfileReport(
    df_raw,
    title='L1 — Profiling UCI Online Retail',
    explorative=True,
    correlations={'pearson': {'calculate': True},
                  'spearman': {'calculate': False},
                  'kendall': {'calculate': False},
                  'phi_k': {'calculate': False}},
    missing_diagrams={'bar': True, 'matrix': True, 'heatmap': True},
    samples={'head': 10, 'tail': 5},
    progress_bar=True
)

# Export HTML
os.makedirs(os.path.dirname(REPORT_OUTPUT), exist_ok=True)
profile.to_file(REPORT_OUTPUT)
print(f' Rapport exporté : {REPORT_OUTPUT}')

# Affichage inline (optionnel, peut être lourd)
# profile.to_notebook_iframe()

 Génération du rapport ydata-profiling...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 10/10 [00:17<00:00,  1.71s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

 Rapport exporté : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Profiling/L1_UCI_profile_report.html


## 8. Synthèse — Décisions de Nettoyage

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║        SYNTHÈSE PROFILING — UCI Online Retail               ║
╠══════════════════════════════════════════════════════════════╣
║ PROBLÈME                      │ ACTION RECOMMANDÉE           ║
╠══════════════════════════════════════════════════════════════╣
║ CustomerID ~25% nulls          │ EXCLURE pour RFM/CLV        ║
║ Factures C* (retours)          │ SÉPARER en fact_returns     ║
║ Codes spéciaux (POST, D, M...) │ FILTRER en Silver           ║
║ Quantity négatifs (hors C*)    │ INVESTIGUER + LOGGER        ║
║ UnitPrice = 0                  │ EXCLURE du calcul revenue   ║
║ Description : nulls + casse    │ STANDARDISER en upper()     ║
║ Doublons stricts               │ DÉDUPLIQUER en Silver       ║
║ Concentration UK (91%)         │ OK — noter dans DECISIONS   ║
╚══════════════════════════════════════════════════════════════╝

→ Documenter chaque décision avec impact chiffré dans DECISIONS.md
→ Perte acceptable cible : < 5% du volume initial
""")

# Résumé chiffré final
summary_final = pd.DataFrame({
    'Règle de nettoyage': [
        'Supprimer doublons stricts',
        'Exclure CustomerID null (RFM scope)',
        'Séparer retours (InvoiceNo C*)',
        'Exclure codes spéciaux (POST, D, C2, M, BANK CHARGES)',
        'Exclure UnitPrice <= 0',
        'Exclure Quantity = 0',
    ],
    'Lignes impactées': [
        df_raw.duplicated().sum(),
        df_raw['CustomerID'].isnull().sum(),
        df_raw['InvoiceNo'].str.startswith('C', na=False).sum(),
        df_raw['StockCode'].isin(SPECIAL_CODES).sum(),
        (df_raw['UnitPrice'] <= 0).sum(),
        (df_raw['Quantity'] == 0).sum(),
    ]
})
summary_final['Pct du total'] = (summary_final['Lignes impactées'] / len(df_raw) * 100).round(2)
display(summary_final.style
    .format({'Lignes impactées': '{:,}', 'Pct du total': '{:.2f}%'})
    .background_gradient(subset=['Pct du total'], cmap='Oranges')
    .set_caption('📋 Récapitulatif des règles de nettoyage Silver'))


╔══════════════════════════════════════════════════════════════╗
║        SYNTHÈSE PROFILING — UCI Online Retail               ║
╠══════════════════════════════════════════════════════════════╣
║ PROBLÈME                      │ ACTION RECOMMANDÉE           ║
╠══════════════════════════════════════════════════════════════╣
║ CustomerID ~25% nulls          │ EXCLURE pour RFM/CLV        ║
║ Factures C* (retours)          │ SÉPARER en fact_returns     ║
║ Codes spéciaux (POST, D, M...) │ FILTRER en Silver           ║
║ Quantity négatifs (hors C*)    │ INVESTIGUER + LOGGER        ║
║ UnitPrice = 0                  │ EXCLURE du calcul revenue   ║
║ Description : nulls + casse    │ STANDARDISER en upper()     ║
║ Doublons stricts               │ DÉDUPLIQUER en Silver       ║
║ Concentration UK (91%)         │ OK — noter dans DECISIONS   ║
╚══════════════════════════════════════════════════════════════╝

→ Documenter chaque décision avec impact chiffré dans DECISIONS.md
→ Perte acceptable cib

,Règle de nettoyage,Lignes impactées,Pct du total
0,Supprimer doublons stricts,"5,268",0.97%
1,Exclure CustomerID null (RFM scope),"135,080",24.93%
2,Séparer retours (InvoiceNo C*),"9,288",1.71%
3,"Exclure codes spéciaux (POST, D, C2, M, BANK CHARGES)","2,845",0.52%
4,Exclure UnitPrice <= 0,"2,517",0.46%
5,Exclure Quantity = 0,0,0.00%


---
## 9. Export des Métriques de Profiling

Sauvegarde d'un fichier JSON de synthèse pour traçabilité dans le pipeline.

In [19]:
import json

metrics = {
    'dataset': 'UCI Online Retail',
    'total_rows': int(total_rows),
    'total_cols': int(total_cols),
    'null_rate_global_pct': round(total_nulls / total_cells * 100, 2),
    'duplicate_rows': int(total_dupes),
    'return_invoices': int(n_returns),
    'unique_customers': int(df_raw['CustomerID'].nunique()),
    'unique_skus': int(df_raw['StockCode'].nunique()),
    'unique_countries': int(df_raw['Country'].nunique()),
    'period_start': str(df_raw['InvoiceDate'].min().date()),
    'period_end': str(df_raw['InvoiceDate'].max().date()),
    'null_by_column': {c: int(df_raw[c].isnull().sum()) for c in df_raw.columns},
}

metrics_path = '../Profiling/L1_UCI_metrics.json'
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f' Métriques exportées : {metrics_path}')
print('\n Notebook L1 UCI — Profiling terminé.')

 Métriques exportées : ../Profiling/L1_UCI_metrics.json

 Notebook L1 UCI — Profiling terminé.
